# Scientific Interest in AMR

This notebook defines **scientific interest** (`SI`) using a **global pooled expected-count baseline** instead of directly reusing `resid_log_bg` from `5_normalisation.ipynb`.

The core idea is:

- `count` is the observed AMR-positive publication count for one `(pathogen, antibiotic, interval)` row.
- `bg_pathogen_n` is the total publication background for the pathogen in the same interval.
- `bg_antibiotic_n` is the total publication background for the antibiotic in the same interval.
- these marginals do **not** contain resistance information by themselves, but they do define a sensible **background-attention baseline**.

So we fit a model for:

`log1p(count)` = intercept + pathogen effect + antibiotic effect + effect of `log1p(bg_pathogen_n)` + effect of `log1p(bg_antibiotic_n)`

This is a **global pooled model with pathogen and antibiotic effects**.

It answers the question:

> given only general publication attention to this pathogen and this antibiotic, what AMR count would be typical?

Then we define:

- `expected_log_count_global`: fitted baseline expected `log1p(count)`
- `expected_count_global`: the same expectation converted back to count scale
- `SI = log1p(observed_count) - expected_log_count_global`

Under this definition:

- positive `SI` means more AMR publications than expected from the pooled background baseline
- negative `SI` means fewer AMR publications than expected

This is different from asking whether marginals can *explain resistance*. They cannot. Instead, the marginals define a **null publication baseline**, and `SI` becomes a residual or surprise score relative to that baseline.

After constructing the pooled baseline, we fit a LOESS smoother directly to `log1p(count)` for one pathogen-antibiotic pair and then express `SI` from that fitted count trajectory.

In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Ridge

from amr.normalisation import BackgroundNormaliseConfig, normalise_by_background

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## Why change the SI definition?

The earlier notebook used `resid_log_bg`, which is based on an interval-relative expected-count construction. That approach is useful descriptively, but for forecasting it has an important limitation: the expected count for one interval depends on how the whole set of intervals for that pair is normalized.

For forward prediction, it is cleaner to use an expected-count model that depends only on quantities that exist at the same time point:

- pathogen background in that interval
- antibiotic background in that interval
- a pooled pathogen effect
- a pooled antibiotic effect

That gives a forward-looking baseline for expected AMR publication volume.

## Chosen baseline model

We use a pooled linear model on log scale:

`log1p(count)` = `alpha` + `u_pathogen` + `v_antibiotic` + `beta_path * log1p(bg_pathogen_n)` + `beta_ab * log1p(bg_antibiotic_n)`

Interpretation:

- `u_pathogen` captures the overall AMR-publication propensity of a pathogen across all antibiotics.
- `v_antibiotic` captures the overall AMR-publication propensity of an antibiotic across all pathogens.
- the two log-background terms capture how general pathogen attention and general antibiotic attention scale AMR publication counts.

## New SI definition

Once the baseline is fitted, the new scientific-interest score is:

`SI = log1p(count) - expected_log_count_global`

That is the observed `SI`. For smoothing and extrapolation we do not fit `SI` directly. Instead we fit a LOESS smoother to `log1p(count)` for the selected pair and derive fitted or predicted `SI` from that count fit:

`predicted_SI = predicted_log_count_loess - expected_log_count_global`

## Why fit counts first

This keeps the workflow close to the question being asked:

- first estimate a smooth trajectory for AMR publication counts
- then measure scientific interest as the gap between that smooth count trajectory and the pooled background expectation

So the model is easier to inspect: the orange curve is the fitted count trajectory, and the red curve is simply the `SI` implied by that orange curve.

## Practical limitation

This notebook still uses interval-level data, not yearly pair-level data. So the LOESS curve is fitted to interval end-years such as `1979, 1984, ..., 2024` and then evaluated on a yearly grid for visualization. The observed points are still the interval-level observations.

In [ ]:
def load_amr_background_df(*, alpha: float = 1.0, use_cached_norm: bool = True, save_cache: bool = True) -> pd.DataFrame:
    """Load the AMR interval data and attach background counts.

    The minimum files needed on disk are:
    - pmids_results_positive.parquet
    - pathogen_tiab_counts.parquet
    - antibiotic_tiab_counts.parquet

    If amr_df_normalised.parquet already exists, reuse it as a cache for the
    background-count columns. Otherwise recompute it with the fixed
    normalisation code.
    """
    cache_path = "amr_df_normalised.parquet"

    if use_cached_norm and os.path.exists(cache_path):
        amr_bg_df = pd.read_parquet(cache_path)
    else:
        amr_df = pd.read_parquet("pmids_results_positive.parquet")
        pa_counts_df = pd.read_parquet("pathogen_tiab_counts.parquet")
        ab_counts_df = pd.read_parquet("antibiotic_tiab_counts.parquet")
        amr_bg_df = normalise_by_background(
            amr_df,
            pathogen_year_df=pa_counts_df,
            antibiotic_year_df=ab_counts_df,
            cfg=BackgroundNormaliseConfig(alpha=alpha),
        )
        if save_cache:
            amr_bg_df.to_parquet(cache_path, index=False)

    amr_bg_df = amr_bg_df.copy()
    amr_bg_df["start_year"] = pd.to_datetime(amr_bg_df["start_date"]).dt.year.astype(int)
    amr_bg_df["end_year"] = pd.to_datetime(amr_bg_df["end_date"]).dt.year.astype(int)
    amr_bg_df["mid_year"] = (amr_bg_df["start_year"] + amr_bg_df["end_year"]) / 2.0
    amr_bg_df["log_count"] = np.log1p(pd.to_numeric(amr_bg_df["count"], errors="coerce"))
    amr_bg_df["log_bg_pathogen_n"] = np.log1p(pd.to_numeric(amr_bg_df["bg_pathogen_n"], errors="coerce"))
    amr_bg_df["log_bg_antibiotic_n"] = np.log1p(pd.to_numeric(amr_bg_df["bg_antibiotic_n"], errors="coerce"))
    amr_bg_df = amr_bg_df.dropna(subset=["log_count", "log_bg_pathogen_n", "log_bg_antibiotic_n"])
    return amr_bg_df


def fit_global_pooled_si_model(amr_bg_df: pd.DataFrame, *, alpha: float = 1.0) -> tuple[pd.DataFrame, Ridge, list[str]]:
    """Fit the pooled expected-count baseline and define SI as its residual.

    The model is a ridge regression on a design matrix containing:
    - log pathogen background count
    - log antibiotic background count
    - one-hot pathogen effects
    - one-hot antibiotic effects

    We use a small amount of regularization because the one-hot design matrix
    is large and some pathogens / antibiotics are sparse.
    """
    work = amr_bg_df.copy()

    pathogen_dummies = pd.get_dummies(work["pathogen"].astype(str), prefix="pathogen", dtype=float)
    antibiotic_dummies = pd.get_dummies(work["antibiotic"].astype(str), prefix="antibiotic", dtype=float)
    x_df = pd.concat(
        [
            work[["log_bg_pathogen_n", "log_bg_antibiotic_n"]].reset_index(drop=True),
            pathogen_dummies.reset_index(drop=True),
            antibiotic_dummies.reset_index(drop=True),
        ],
        axis=1,
    )
    y = work["log_count"].to_numpy(dtype=float)

    model = Ridge(alpha=alpha, fit_intercept=True)
    model.fit(x_df, y)

    work["expected_log_count_global"] = model.predict(x_df)
    work["expected_count_global"] = np.clip(np.expm1(work["expected_log_count_global"]), 0.0, None)
    work["SI"] = work["log_count"] - work["expected_log_count_global"]
    return work, model, x_df.columns.tolist()


amr_bg_df = load_amr_background_df()
si_df, pooled_model, pooled_feature_columns = fit_global_pooled_si_model(amr_bg_df, alpha=1.0)

display(
    si_df[
        [
            "pathogen",
            "antibiotic",
            "start_year",
            "end_year",
            "count",
            "bg_pathogen_n",
            "bg_antibiotic_n",
            "expected_count_global",
            "SI",
        ]
    ].head()
)

## Inspect the pooled baseline

A few quantities are worth checking after the pooled model is fitted:

- `expected_count_global`: the background-only expected AMR count
- `SI`: the surprise or residual score on log scale
- the coefficients on `log_bg_pathogen_n` and `log_bg_antibiotic_n`

The one-hot effects are numerous, so this notebook does not print all of them by default. But the two continuous coefficients are often informative: they show how strongly expected AMR volume scales with general pathogen attention and general antibiotic attention.

In [ ]:
coef_df = pd.DataFrame(
    {
        "feature": pooled_feature_columns,
        "coefficient": pooled_model.coef_,
    }
)

display(coef_df[coef_df["feature"].isin(["log_bg_pathogen_n", "log_bg_antibiotic_n"])])

print("Model intercept:", float(pooled_model.intercept_))

## LOESS fit on log-counts

The previous notebook versions tried to forecast AMR counts with several GP-based decompositions. In practice that became too fragile for these interval data: each pair has only a small number of 5-year observations, so the extra layers were hard to identify and the orange curve was mostly reflecting model assumptions rather than signal.

This version uses a simpler idea.

For one pathogen-antibiotic pair we:

1. keep the pooled global baseline `expected_log_count_global`
2. fit a LOESS smoother directly to the observed pair trajectory `log1p(count)` over time
3. use that fitted log-count trajectory as the smooth estimate of future AMR count
4. define fitted or predicted `SI` from counts as:
   `predicted_SI = predicted_log_count_loess - expected_log_count_baseline`

That means the red curve is the `SI` implied by the LOESS fit on log-counts.

### Why this is simpler

The LOESS model is intentionally modest. It does not try to separate trend, residual GPs, future background models, or several interacting uncertainty layers. It only answers:

`What smooth log-count trajectory best matches this pair in the chosen fit window?`

Once we have that trajectory, we compare it against the pooled background expectation. The gap is the fitted `SI`.

### What the uncertainty band means now

The `±1 std` band is based on the residual spread of the training log-counts around the LOESS fit. So it is a simple empirical band, not a formal predictive interval.

### Practical interpretation

- orange line: LOESS fit of `log1p(count)`, converted back to counts
- red line: `SI` implied by that LOESS count fit relative to the pooled baseline
- navy line: observed `SI`
- bars: observed per-interval AMR count and the two marginals on the secondary axis

This keeps the model aligned with the user-facing question: first smooth the counts, then express scientific interest from the smoothed counts.


In [ ]:
def get_pair_si_series(si_df: pd.DataFrame, pathogen: str, antibiotic: str) -> pd.DataFrame:
    pair_df = (
        si_df.loc[
            (si_df["pathogen"] == pathogen) & (si_df["antibiotic"] == antibiotic),
            [
                "pathogen",
                "antibiotic",
                "start_date",
                "end_date",
                "start_year",
                "end_year",
                "mid_year",
                "count",
                "bg_pathogen_n",
                "bg_antibiotic_n",
                "expected_count_global",
                "expected_log_count_global",
                "SI",
            ],
        ]
        .sort_values(["end_year", "start_year"])
        .reset_index(drop=True)
    )

    if pair_df.empty:
        raise ValueError(f"No SI data found for pathogen={pathogen!r}, antibiotic={antibiotic!r}.")

    pair_df = (
        pair_df.groupby(["pathogen", "antibiotic", "start_year", "end_year", "mid_year"], as_index=False)
        .agg(
            {
                "count": "sum",
                "bg_pathogen_n": "max",
                "bg_antibiotic_n": "max",
                "expected_count_global": "sum",
                "expected_log_count_global": "mean",
                "SI": "mean",
            }
        )
        .sort_values("end_year")
        .reset_index(drop=True)
    )
    pair_df["log_count"] = np.log1p(pair_df["count"].astype(float))
    return pair_df


def _interpolate_expected_log_count_baseline(pair_df: pd.DataFrame, years: np.ndarray) -> np.ndarray:
    """Interpolate the pooled expected log-count baseline onto a yearly grid."""
    anchor_years = pair_df["end_year"].astype(int).to_numpy()
    anchor_vals = pair_df["expected_log_count_global"].astype(float).to_numpy()
    years = np.asarray(years, dtype=float)
    if anchor_years.size == 0:
        return np.zeros_like(years, dtype=float)
    if anchor_years.size == 1:
        return np.full_like(years, float(anchor_vals[0]), dtype=float)
    return np.interp(years, anchor_years.astype(float), anchor_vals, left=anchor_vals[0], right=anchor_vals[-1])


def _loess_predict(train_years: np.ndarray, train_values: np.ndarray, query_years: np.ndarray, *, frac: float) -> np.ndarray:
    train_years = np.asarray(train_years, dtype=float)
    train_values = np.asarray(train_values, dtype=float)
    query_years = np.asarray(query_years, dtype=float)
    if train_years.size == 0:
        raise ValueError("Cannot evaluate LOESS with no training points.")
    if train_years.size == 1:
        return np.full_like(query_years, float(train_values[0]), dtype=float)

    fitted = lowess(
        endog=train_values,
        exog=train_years,
        frac=float(frac),
        it=0,
        xvals=query_years,
        return_sorted=False,
    )
    return np.asarray(fitted, dtype=float)


def _fit_loess_log_count(years: np.ndarray, log_counts: np.ndarray, *, loess_frac: float) -> dict:
    years = np.asarray(years, dtype=float)
    log_counts = np.asarray(log_counts, dtype=float)
    if years.size == 0:
        raise ValueError("Cannot fit LOESS with no training points.")
    if years.size == 1:
        constant_level = float(log_counts[0])
        return {
            "frac": float(loess_frac),
            "train_fitted_log_count": np.full_like(log_counts, constant_level, dtype=float),
            "train_residuals": np.zeros_like(log_counts, dtype=float),
            "residual_std": 0.0,
            "constant_level": constant_level,
            "left_slope": 0.0,
            "right_slope": 0.0,
            "year_min": float(years[0]),
            "year_max": float(years[0]),
            "left_value": constant_level,
            "right_value": constant_level,
            "train_years": years,
            "train_log_counts": log_counts,
        }

    fitted_log_counts = _loess_predict(years, log_counts, years, frac=loess_frac)
    residuals = log_counts - fitted_log_counts
    residual_std = float(np.std(residuals, ddof=1)) if residuals.size > 1 else 0.0
    if residual_std < 1e-8 and log_counts.size > 1:
        residual_std = float(0.25 * np.std(np.diff(log_counts), ddof=1)) if log_counts.size > 2 else float(0.1 * abs(log_counts[-1] - log_counts[0]))
    left_slope = float((log_counts[1] - log_counts[0]) / (years[1] - years[0]))
    right_slope = float((log_counts[-1] - log_counts[-2]) / (years[-1] - years[-2]))
    return {
        "frac": float(loess_frac),
        "train_fitted_log_count": fitted_log_counts,
        "train_residuals": residuals,
        "residual_std": residual_std,
        "constant_level": None,
        "left_slope": left_slope,
        "right_slope": right_slope,
        "year_min": float(years[0]),
        "year_max": float(years[-1]),
        "left_value": float(log_counts[0]),
        "right_value": float(log_counts[-1]),
        "train_years": years,
        "train_log_counts": log_counts,
    }


def _evaluate_loess_with_linear_tails(loess_fit: dict, years: np.ndarray) -> np.ndarray:
    years = np.asarray(years, dtype=float)
    if loess_fit["constant_level"] is not None:
        return np.full_like(years, loess_fit["constant_level"], dtype=float)
    pred = np.empty_like(years, dtype=float)
    inside_mask = (years >= loess_fit["year_min"]) & (years <= loess_fit["year_max"])
    left_mask = years < loess_fit["year_min"]
    right_mask = years > loess_fit["year_max"]
    pred[inside_mask] = _loess_predict(
        loess_fit["train_years"],
        loess_fit["train_log_counts"],
        years[inside_mask],
        frac=loess_fit["frac"],
    )
    pred[left_mask] = loess_fit["left_value"] + loess_fit["left_slope"] * (years[left_mask] - loess_fit["year_min"])
    pred[right_mask] = loess_fit["right_value"] + loess_fit["right_slope"] * (years[right_mask] - loess_fit["year_max"])
    return pred


def fit_predict_si_gp(
    si_df: pd.DataFrame,
    pathogen: str,
    antibiotic: str,
    fit_start_year: int,
    fit_end_year: int,
    num_additional_years: int = 5,
    *,
    min_train_points: int = 3,
    loess_frac: float = 0.6,
) -> dict:
    if fit_end_year < fit_start_year:
        raise ValueError("fit_end_year must be >= fit_start_year.")
    if num_additional_years < 0:
        raise ValueError("num_additional_years must be >= 0.")

    pair_df = get_pair_si_series(si_df, pathogen, antibiotic)
    latest_available_year = int(pair_df["end_year"].max())

    train_df = pair_df.loc[pair_df["end_year"].between(int(fit_start_year), int(fit_end_year))].copy()
    if train_df["end_year"].nunique() < min_train_points:
        raise ValueError(
            f"Need at least {min_train_points} observed intervals between {fit_start_year} and {fit_end_year}; "
            f"found {train_df['end_year'].nunique()}."
        )

    loess_fit = _fit_loess_log_count(
        train_df["end_year"].to_numpy(dtype=float),
        train_df["log_count"].to_numpy(dtype=float),
        loess_frac=loess_frac,
    )

    prediction_end_year = latest_available_year + int(num_additional_years)
    model_plot_start_year = int(pair_df["end_year"].min())
    pred_years = np.arange(model_plot_start_year, prediction_end_year + 1, dtype=float)

    pred_df = pd.DataFrame({"year": pred_years.astype(int)})
    pred_df["expected_log_count_baseline"] = _interpolate_expected_log_count_baseline(pair_df, pred_years)
    pred_df["expected_count_baseline"] = np.clip(np.expm1(pred_df["expected_log_count_baseline"]), 0.0, None)

    pred_log_count = _evaluate_loess_with_linear_tails(loess_fit, pred_years)
    pred_df["predicted_log_count_loess"] = pred_log_count
    pred_df["predicted_log_count_std"] = float(loess_fit["residual_std"])
    pred_df["predicted_log_count_loess_lower_1std"] = pred_df["predicted_log_count_loess"] - pred_df["predicted_log_count_std"]
    pred_df["predicted_log_count_loess_upper_1std"] = pred_df["predicted_log_count_loess"] + pred_df["predicted_log_count_std"]
    pred_df["predicted_count_loess"] = np.clip(np.expm1(pred_df["predicted_log_count_loess"]), 0.0, None)
    pred_df["predicted_count_loess_lower_1std"] = np.clip(
        np.expm1(pred_df["predicted_log_count_loess_lower_1std"]),
        0.0,
        None,
    )
    pred_df["predicted_count_loess_upper_1std"] = np.clip(
        np.expm1(pred_df["predicted_log_count_loess_upper_1std"]),
        0.0,
        None,
    )

    pred_df["predicted_SI"] = pred_df["predicted_log_count_loess"] - pred_df["expected_log_count_baseline"]
    pred_df["predicted_std"] = float(loess_fit["residual_std"])
    pred_df["lower_1std"] = pred_df["predicted_SI"] - pred_df["predicted_std"]
    pred_df["upper_1std"] = pred_df["predicted_SI"] + pred_df["predicted_std"]

    observed_window_df = pair_df.loc[
        pair_df["end_year"] >= int(fit_start_year),
        ["end_year", "SI", "count", "log_count", "expected_count_global", "expected_log_count_global"],
    ].copy()
    observed_window_df = observed_window_df.rename(
        columns={
            "end_year": "year",
            "SI": "observed_SI",
            "count": "observed_count",
            "log_count": "observed_log_count",
        }
    )
    comparison_df = observed_window_df.merge(pred_df, on="year", how="left")
    comparison_df["prediction_error"] = comparison_df["observed_SI"] - comparison_df["predicted_SI"]
    comparison_df["abs_error"] = comparison_df["prediction_error"].abs()
    comparison_df["within_1std"] = (
        (comparison_df["observed_SI"] >= comparison_df["lower_1std"])
        & (comparison_df["observed_SI"] <= comparison_df["upper_1std"])
    )
    comparison_df["count_prediction_error_loess"] = comparison_df["observed_count"] - comparison_df["predicted_count_loess"]
    comparison_df["count_abs_error_loess"] = comparison_df["count_prediction_error_loess"].abs()

    return {
        "pathogen": pathogen,
        "antibiotic": antibiotic,
        "fit_start_year": int(fit_start_year),
        "fit_end_year": int(fit_end_year),
        "latest_available_year": latest_available_year,
        "num_additional_years": int(num_additional_years),
        "model_plot_start_year": model_plot_start_year,
        "pair_df": pair_df,
        "train_df": train_df,
        "pred_df": pred_df,
        "comparison_df": comparison_df,
        "loess_fit": loess_fit,
    }


def forecast_si_for_all_pairs(
    si_df: pd.DataFrame,
    fit_start_year: int = 1970,
    fit_end_year: int = 2015,
    num_additional_years: int = 5,
    *,
    current_year: int | None = None,
    min_train_points: int = 3,
    loess_frac: float = 0.6,
    save_path: str | None = None,
) -> pd.DataFrame:
    if current_year is None:
        current_year = int(pd.Timestamp.today().year)
    target_year = int(current_year) + int(num_additional_years)

    pair_index_df = (
        si_df[["pathogen", "antibiotic"]]
        .drop_duplicates()
        .sort_values(["pathogen", "antibiotic"])
        .reset_index(drop=True)
    )

    rows = []
    for pair in pair_index_df.itertuples(index=False):
        pathogen = str(pair.pathogen)
        antibiotic = str(pair.antibiotic)
        try:
            result = fit_predict_si_gp(
                si_df=si_df,
                pathogen=pathogen,
                antibiotic=antibiotic,
                fit_start_year=fit_start_year,
                fit_end_year=fit_end_year,
                num_additional_years=max(num_additional_years, target_year - fit_end_year),
                min_train_points=min_train_points,
                loess_frac=loess_frac,
            )
            target_row = result["pred_df"].loc[result["pred_df"]["year"] == int(target_year)].iloc[0]
            pair_df = result["pair_df"]
            train_df = result["train_df"]
            rows.append(
                {
                    "pathogen": pathogen,
                    "antibiotic": antibiotic,
                    "fit_start_year": int(fit_start_year),
                    "fit_end_year": int(fit_end_year),
                    "current_year": int(current_year),
                    "forecast_year": int(target_year),
                    "num_additional_years": int(num_additional_years),
                    "loess_frac": float(result["loess_fit"]["frac"]),
                    "predicted_SI_mean": float(target_row["predicted_SI"]),
                    "predicted_SI_std": float(target_row["predicted_std"]),
                    "predicted_SI_lower_1std": float(target_row["lower_1std"]),
                    "predicted_SI_upper_1std": float(target_row["upper_1std"]),
                    "predicted_count_mean": float(target_row["predicted_count_loess"]),
                    "predicted_count_lower_1std": float(target_row["predicted_count_loess_lower_1std"]),
                    "predicted_count_upper_1std": float(target_row["predicted_count_loess_upper_1std"]),
                    "latest_available_year": int(result["latest_available_year"]),
                    "pair_first_year": int(pair_df["end_year"].min()),
                    "pair_last_year": int(pair_df["end_year"].max()),
                    "n_pair_intervals": int(pair_df["end_year"].nunique()),
                    "n_train_intervals": int(train_df["end_year"].nunique()),
                    "amr_pair_total_count": float(pair_df["count"].sum()),
                    "status": "ok",
                    "error": None,
                }
            )
        except Exception as exc:
            rows.append(
                {
                    "pathogen": pathogen,
                    "antibiotic": antibiotic,
                    "fit_start_year": int(fit_start_year),
                    "fit_end_year": int(fit_end_year),
                    "current_year": int(current_year),
                    "forecast_year": int(target_year),
                    "num_additional_years": int(num_additional_years),
                    "loess_frac": float(loess_frac),
                    "predicted_SI_mean": np.nan,
                    "predicted_SI_std": np.nan,
                    "predicted_SI_lower_1std": np.nan,
                    "predicted_SI_upper_1std": np.nan,
                    "predicted_count_mean": np.nan,
                    "predicted_count_lower_1std": np.nan,
                    "predicted_count_upper_1std": np.nan,
                    "latest_available_year": np.nan,
                    "pair_first_year": np.nan,
                    "pair_last_year": np.nan,
                    "n_pair_intervals": np.nan,
                    "n_train_intervals": np.nan,
                    "amr_pair_total_count": np.nan,
                    "status": "failed",
                    "error": str(exc),
                }
            )

    forecast_df = pd.DataFrame(rows).sort_values(
        ["status", "predicted_SI_mean", "amr_pair_total_count"],
        ascending=[True, False, False],
        na_position="last",
    ).reset_index(drop=True)

    if save_path is None:
        save_path = (
            f"si_pair_forecasts_fit_{int(fit_start_year)}_{int(fit_end_year)}"
            f"_target_{int(target_year)}.parquet"
        )
    forecast_df.to_parquet(save_path, index=False)
    return forecast_df


def plot_si_prediction(
    si_df: pd.DataFrame,
    pathogen: str,
    antibiotic: str,
    fit_start_year: int,
    fit_end_year: int,
    num_additional_years: int = 5,
    *,
    figsize: tuple[int, int] = (12, 6),
    loess_frac: float = 0.6,
) -> dict:
    result = fit_predict_si_gp(
        si_df=si_df,
        pathogen=pathogen,
        antibiotic=antibiotic,
        fit_start_year=fit_start_year,
        fit_end_year=fit_end_year,
        num_additional_years=num_additional_years,
        loess_frac=loess_frac,
    )

    pair_df = result["pair_df"]
    pred_df = result["pred_df"]

    fig, ax = plt.subplots(figsize=figsize)
    ax2 = ax.twinx()

    count_plot_df = pair_df[["end_year", "count", "bg_pathogen_n", "bg_antibiotic_n"]].copy()
    count_plot_df["count_plot"] = np.clip(count_plot_df["count"].astype(float), 1.0, None)
    count_plot_df["bg_pathogen_plot"] = np.clip(count_plot_df["bg_pathogen_n"].astype(float), 1.0, None)
    count_plot_df["bg_antibiotic_plot"] = np.clip(count_plot_df["bg_antibiotic_n"].astype(float), 1.0, None)
    bar_width = 1.15
    ax2.bar(
        count_plot_df["end_year"] - bar_width,
        count_plot_df["count_plot"],
        width=bar_width,
        color="lightskyblue",
        alpha=0.4,
        label="Per-interval AMR count",
        zorder=0,
    )
    ax2.bar(
        count_plot_df["end_year"],
        count_plot_df["bg_pathogen_plot"],
        width=bar_width,
        color="tab:green",
        alpha=0.28,
        label="Pathogen background count",
        zorder=0,
    )
    ax2.bar(
        count_plot_df["end_year"] + bar_width,
        count_plot_df["bg_antibiotic_plot"],
        width=bar_width,
        color="tab:purple",
        alpha=0.22,
        label="Antibiotic background count",
        zorder=0,
    )
    ax2.set_yscale("log")
    ax2.set_ylabel("Publication counts (log scale)", color="tab:blue")
    ax2.tick_params(axis="y", labelcolor="tab:blue")

    observed_fit_df = pair_df.loc[pair_df["end_year"] <= int(fit_end_year)].copy()
    observed_future_df = pair_df.loc[pair_df["end_year"] > int(fit_end_year)].copy()
    if not observed_fit_df.empty and not observed_future_df.empty:
        observed_future_df = pd.concat([observed_fit_df.tail(1), observed_future_df], ignore_index=True)

    ax.plot(
        observed_fit_df["end_year"],
        observed_fit_df["SI"],
        color="navy",
        marker="o",
        linewidth=2.6,
        markersize=8,
        markerfacecolor="white",
        markeredgecolor="navy",
        markeredgewidth=2.0,
        label="Observed SI (fit region)",
        zorder=4,
    )
    if not observed_future_df.empty:
        ax.plot(
            observed_future_df["end_year"],
            observed_future_df["SI"],
            color="navy",
            marker="o",
            linewidth=2.6,
            linestyle="--",
            markersize=8,
            markerfacecolor="white",
            markeredgecolor="navy",
            markeredgewidth=2.0,
            label="Observed SI (after fit window)",
            zorder=4,
        )

    fit_pred_df = pred_df.loc[pred_df["year"] <= int(fit_end_year)].copy()
    forecast_pred_df = pred_df.loc[pred_df["year"] >= int(fit_end_year)].copy()

    ax.axvspan(pair_df["end_year"].min(), fit_end_year, color="tab:blue", alpha=0.06, zorder=1)
    ax.axvline(fit_end_year, color="tab:blue", linestyle="--", linewidth=1.6, alpha=0.75, zorder=2)
    ax.text(
        fit_end_year + 0.2,
        0.97,
        f"fit end = {fit_end_year}",
        transform=ax.get_xaxis_transform(),
        color="tab:blue",
        fontsize=11,
    )

    ax.plot(
        fit_pred_df["year"],
        fit_pred_df["predicted_SI"],
        color="tab:red",
        linewidth=2.4,
        label="LOESS SI (fit region)",
        zorder=3,
    )
    ax.plot(
        forecast_pred_df["year"],
        forecast_pred_df["predicted_SI"],
        color="tab:red",
        linewidth=2.4,
        linestyle="--",
        label="LOESS SI (forecast region)",
        zorder=3,
    )
    ax.fill_between(
        pred_df["year"],
        pred_df["lower_1std"],
        pred_df["upper_1std"],
        color="tab:red",
        alpha=0.14,
        label="LOESS SI ±1 std",
        zorder=2,
    )

    ax2.plot(
        fit_pred_df["year"],
        np.clip(fit_pred_df["predicted_count_loess"], 1.0, None),
        color="darkorange",
        linewidth=2.4,
        label="LOESS AMR count (fit region)",
        zorder=2,
    )
    ax2.plot(
        forecast_pred_df["year"],
        np.clip(forecast_pred_df["predicted_count_loess"], 1.0, None),
        color="darkorange",
        linewidth=2.4,
        linestyle="--",
        label="LOESS AMR count (forecast region)",
        zorder=2,
    )
    ax2.fill_between(
        pred_df["year"],
        np.clip(pred_df["predicted_count_loess_lower_1std"], 1.0, None),
        np.clip(pred_df["predicted_count_loess_upper_1std"], 1.0, None),
        color="darkorange",
        alpha=0.08,
        label="LOESS AMR count ±1 std",
        zorder=1,
    )

    ax.set_xlabel("Interval end year")
    ax.set_ylabel("SI = fitted log-count - pooled expected log-count")
    ax.set_title(f"Scientific Interest (SI) LOESS count fit: {pathogen} vs {antibiotic}")
    ax.grid(True, axis="both", alpha=0.2)

    handles1, labels1 = ax.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(
        handles1 + handles2,
        labels1 + labels2,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=4,
        frameon=False,
    )
    plt.tight_layout(rect=(0, 0.08, 1, 1))
    plt.show()

    return result


## Example

The example below fits a LOESS smoother directly to `log1p(count)` for one pair, converts that smooth count trajectory back to counts, and then expresses `SI` as the gap between the fitted log-count and the pooled global baseline.

So the model interpretation is now:

- first smooth the AMR counts on the log scale
- then derive `SI` from those fitted counts

This keeps the workflow simple and makes the red `SI` curve a direct consequence of the orange count fit.


In [ ]:
result = plot_si_prediction(
    si_df=si_df,
    pathogen="Escherichia coli",
    antibiotic="Ampicillin",
    fit_start_year=1994,
    fit_end_year=2015,
    num_additional_years=10,
    loess_frac=0.6,
)

display(result["comparison_df"])
print("LOESS fraction:", result["loess_fit"]["frac"])
print("Training residual std on log-count scale:", result["loess_fit"]["residual_std"])
print("Latest observed interval end year:", result["latest_available_year"])
